# 03 — Risk workflow_update

**NON_BASELINE_RUN**. Prepare top-10/20-bit, then rerank exact/QAOA artifacts if available.

In [ ]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent
BASE = [
    "--config",
    "configs/base.yaml",
    "--profile",
    "configs/profiles/workflow_update.yaml",
    "--override",
    "configs/provisional/workflow_update_downstream.yaml",
]


def run(name, cmd):
    print("$", " ".join(cmd), flush=True)
    t = time.perf_counter()
    p = subprocess.run(cmd, cwd=ROOT, check=False)
    elapsed = time.perf_counter() - t
    print(f"[{name}] exit={p.returncode} elapsed={elapsed:.2f}s")
    if p.returncode:
        raise RuntimeError(f"{name} failed")
    return elapsed

In [ ]:
prepare_s = run(
    "prepare-workflow", ["uv", "run", "qshield-risk", "prepare-workflow", *BASE]
)

In [ ]:
from pathlib import Path

q = ROOT / "artifacts/dev/optimization/qaoa_results.json"
if q.exists():
    rerank_s = run(
        "rerank-polish", ["uv", "run", "qshield-risk", "rerank-polish", *BASE]
    )
    true_s = run(
        "benchmark-true", ["uv", "run", "qshield-risk", "benchmark-true", *BASE]
    )
else:
    print("SKIP rerank/benchmark-true: quantum artifact missing")
    rerank_s = true_s = None
print("timing", {"prepare": prepare_s, "rerank": rerank_s, "benchmark_true": true_s})
checks = [
    ROOT / "artifacts/dev/risk/candidate_top10.csv",
    ROOT / "artifacts/dev/risk/candidate_order.json",
    ROOT / "artifacts/dev/risk/final_recommendation.json",
    ROOT / "artifacts/dev/risk/true_benchmark.json",
]
for p in checks:
    print(("OK" if p.exists() else "MISSING"), p)